# Lookahead Logit Acceleration — MT-Bench Benchmark

Tests whether the free lookahead logits from a single autoregressive forward pass
can serve as exact draft tokens for speculative decoding — at zero extra cost.

**Recommended runtimes:**
- G4 (L40S 48GB) → 32B model, bfloat16
- H100 (80GB) → 32B model, ~2x faster
- A100 (40GB) → 14B model
- T4 (16GB) → 1.5B model

**Quick start:** Runtime → Change runtime type → G4 GPU, then run all cells.

In [ ]:
# Cell 1: Install dependencies
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'accelerate', 'torch'])

# Flash attention (optional, significant speedup on Ampere+)
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'flash-attn', '--no-build-isolation'])
    FLASH_ATTN = True
except Exception:
    print('flash-attn not available, using eager attention')
    FLASH_ATTN = False

print('Dependencies installed.')

In [ ]:
# Cell 2: Imports + GPU check
import json, time
from collections import defaultdict

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}  ({mem:.0f} GB)')
else:
    print('No GPU detected — will run on CPU (slow for large models)')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Cell 3: Configuration
# @param {type:"string"}
RUN_MODE = 'quick_test'   # 'quick_test' or 'full'

# @param ["0.5B", "1.5B", "7B", "14B", "32B"]
MODEL_SIZE = '32B'

MAX_NEW_TOKENS = 512

MODEL_MAP = {
    '0.5B': 'Qwen/Qwen2.5-0.5B-Instruct',
    '1.5B': 'Qwen/Qwen2.5-1.5B-Instruct',
    '7B':   'Qwen/Qwen2.5-7B-Instruct',
    '14B':  'Qwen/Qwen2.5-14B-Instruct',
    '32B':  'Qwen/Qwen2.5-32B-Instruct',
}

MODEL_ID = MODEL_MAP[MODEL_SIZE]
NUM_QUESTIONS = 1 if RUN_MODE == 'quick_test' else 65

print(f'Model:      {MODEL_ID}')
print(f'Mode:       {RUN_MODE} ({NUM_QUESTIONS} question(s))')
print(f'Max tokens: {MAX_NEW_TOKENS}')

In [ ]:
# Cell 4: MT-Bench questions (65 questions, 8 categories)
MT_BENCH_QUESTIONS = [
    # writing
    {'id': 81,  'category': 'writing',    'turn': 'Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.'},
    {'id': 82,  'category': 'writing',    'turn': "Draft a professional email seeking your supervisor's feedback on the 'Quarterly Financial Report' you prepared."},
    {'id': 83,  'category': 'writing',    'turn': "Imagine you are writing a blog post comparing two popular smartphone models. Develop an outline for the blog post, including key points and subheadings."},
    {'id': 84,  'category': 'writing',    'turn': 'Write a persuasive essay arguing that the government should invest more in renewable energy sources.'},
    {'id': 85,  'category': 'writing',    'turn': "Write a 300-word review for a local restaurant called 'The Cozy Nook', focusing on ambiance, service, and food quality."},
    {'id': 86,  'category': 'writing',    'turn': 'Compose a haiku sequence that captures the essence of each of the four seasons.'},
    {'id': 87,  'category': 'writing',    'turn': 'Write a short story about a robot that discovers music for the first time.'},
    {'id': 88,  'category': 'writing',    'turn': 'Craft a poem about the journey of a single raindrop from a cloud to the ocean.'},
    {'id': 89,  'category': 'writing',    'turn': 'Write a marketing pitch for a new eco-friendly water bottle targeting environmentally conscious millennials.'},
    {'id': 90,  'category': 'writing',    'turn': 'Create a short bedtime story suitable for five-year-olds about a young rabbit who learns the importance of sharing.'},
    # roleplay
    {'id': 101, 'category': 'roleplay',   'turn': 'Imagine you are an experienced Michelin-star chef. Provide a detailed recipe for a classic French dish, explaining the technique and the importance of each ingredient.'},
    {'id': 102, 'category': 'roleplay',   'turn': 'You are an astronaut describing your first experience walking on the Moon. Share your feelings and observations in detail.'},
    {'id': 103, 'category': 'roleplay',   'turn': 'Act as a math teacher and explain the Pythagorean theorem to a 10-year-old student.'},
    {'id': 104, 'category': 'roleplay',   'turn': 'You are a time-traveling historian who has just returned from ancient Rome. Describe your experience.'},
    {'id': 105, 'category': 'roleplay',   'turn': 'As a renowned chef, you are judging a cooking competition. Describe a dish presented to you and provide your critique.'},
    {'id': 106, 'category': 'roleplay',   'turn': "You are a cybersecurity expert. Explain what a SQL injection attack is, how it works, and how to prevent it."},
    {'id': 107, 'category': 'roleplay',   'turn': 'You are a virtual assistant designed to help elderly users. Explain how to use a smartphone to make a video call.'},
    {'id': 108, 'category': 'roleplay',   'turn': 'As a personal financial advisor, explain the concept of compound interest and how it can be used to build wealth over time.'},
    {'id': 109, 'category': 'roleplay',   'turn': 'You are a nutritionist. Create a healthy one-week meal plan for someone who wants to lose weight but dislikes vegetables.'},
    {'id': 110, 'category': 'roleplay',   'turn': 'As a travel agent, suggest a 10-day itinerary for a family trip to Japan, including cultural experiences and must-see attractions.'},
    # reasoning
    {'id': 111, 'category': 'reasoning',  'turn': 'The vertices of a triangle are at points (0, 0), (-1, 1), and (3, 3). What is the area of the triangle?'},
    {'id': 112, 'category': 'reasoning',  'turn': 'A group of four people needs to cross a bridge at night. They have only one torch and the bridge can support only two people at a time. Person A can cross in 1 minute, B in 2 minutes, C in 5 minutes, and D in 8 minutes. When two people cross together, they move at the slower person\u2019s pace. What is the minimum time for all of them to cross the bridge?'},
    {'id': 113, 'category': 'reasoning',  'turn': "In a town, there are three types of people: truth-tellers (always tell the truth), liars (always lie), and alternators (alternate between truth and lie). You meet three people: A, B, and C. A says 'B is a liar.' B says 'C is an alternator.' C says 'A is a truth-teller.' Determine the type of each person."},
    {'id': 114, 'category': 'reasoning',  'turn': 'You have 8 identical-looking balls, one of which is slightly heavier than the others. Using a balance scale and only two weighings, how can you identify the heavier ball?'},
    {'id': 115, 'category': 'reasoning',  'turn': 'A farmer has a rectangular field with a perimeter of 400 meters. He wants to divide the field into two equal halves by building a fence parallel to one side. If the total length of the fence (including the dividing fence) is 500 meters, what are the dimensions of the field?'},
    {'id': 116, 'category': 'reasoning',  'turn': 'In a chess tournament, each player plays every other player exactly once. If there are 10 players in the tournament, how many games are played in total?'},
    {'id': 117, 'category': 'reasoning',  'turn': 'A train travels from station A to station B at 60 km/h and returns at 90 km/h. What is the average speed for the entire trip?'},
    {'id': 118, 'category': 'reasoning',  'turn': 'A box contains 5 red balls and 5 blue balls. If you draw 3 balls at random without replacement, what is the probability that you draw at least 2 red balls?'},
    {'id': 119, 'category': 'reasoning',  'turn': "A clock's minute hand is 10 cm long. How far does the tip of the minute hand travel in 35 minutes?"},
    {'id': 120, 'category': 'reasoning',  'turn': "In a particular language, 'mab kun pala' means 'sky is blue', 'kun pala vertu' means 'is blue beautiful', and 'mab vertu pala' means 'sky beautiful blue'. What does each individual word mean?"},
    # math
    {'id': 121, 'category': 'math',       'turn': 'Solve for x in the equation: 2x + 5 = 3x - 7.'},
    {'id': 122, 'category': 'math',       'turn': 'If a triangle has sides of length 3, 4, and 5, what is its area?'},
    {'id': 123, 'category': 'math',       'turn': 'Find the derivative of f(x) = x^3 - 2x^2 + x - 5.'},
    {'id': 124, 'category': 'math',       'turn': 'What is the integral of f(x) = 3x^2 + 2x - 1 from x = 0 to x = 3?'},
    {'id': 125, 'category': 'math',       'turn': 'Solve the following system of equations: 2x + 3y = 11 and 4x - y = 5.'},
    {'id': 126, 'category': 'math',       'turn': 'A series is defined by the recursive formula a(n) = 2 * a(n-1) + 1, with a(1) = 1. What is a(10)?'},
    {'id': 127, 'category': 'math',       'turn': 'If log(x) = 2, what is the value of log(x^3)?'},
    {'id': 128, 'category': 'math',       'turn': 'The probability of rain on any given day in April is 0.3. What is the probability that it will rain on exactly 5 days in April?'},
    {'id': 129, 'category': 'math',       'turn': 'In a geometric sequence, the first term is 2 and the common ratio is 3. What is the sum of the first 6 terms?'},
    {'id': 130, 'category': 'math',       'turn': 'Find the minimum value of the function f(x) = 3x^2 - 6x + 4 using the completing the square method.'},
    # coding
    {'id': 131, 'category': 'coding',     'turn': 'Write a Python function to find the longest common prefix string amongst an array of strings.'},
    {'id': 132, 'category': 'coding',     'turn': 'Implement a Python class for a stack data structure with push, pop, and peek methods.'},
    {'id': 133, 'category': 'coding',     'turn': 'Write a Python function to check if a string is a palindrome, ignoring case and non-alphanumeric characters.'},
    {'id': 134, 'category': 'coding',     'turn': 'Implement a Python function to find all prime numbers up to a given number using the Sieve of Eratosthenes algorithm.'},
    {'id': 135, 'category': 'coding',     'turn': 'Write a Python function to merge two sorted arrays into a single sorted array.'},
    {'id': 136, 'category': 'coding',     'turn': "Write a function to find the maximum subarray sum using Kadane's algorithm."},
    {'id': 137, 'category': 'coding',     'turn': 'Implement binary search in Python. The function should return the index of the target if found, or -1 if not found.'},
    {'id': 138, 'category': 'coding',     'turn': 'Write a Python function to convert a Roman numeral string to an integer.'},
    {'id': 139, 'category': 'coding',     'turn': 'Write a Python function that takes a list of integers and returns a new list with duplicates removed, preserving order.'},
    {'id': 140, 'category': 'coding',     'turn': 'Implement a Python function to flatten a nested list of arbitrary depth.'},
    # extraction
    {'id': 141, 'category': 'extraction', 'turn': "Given the following paragraph, identify all the named entities (persons, organizations, locations) mentioned: 'Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in Cupertino, California in 1976.'"},
    {'id': 142, 'category': 'extraction', 'turn': "Extract all email addresses from the following text: 'Contact us at support@example.com or sales@company.org for more information. You can also reach our CEO at john.doe@enterprise.net.'"},
    {'id': 143, 'category': 'extraction', 'turn': "Extract all dates mentioned in the following text and convert them to the format YYYY-MM-DD: 'The project started on January 5th, 2023, and is expected to end by the 15th of March, 2024.'"},
    {'id': 144, 'category': 'extraction', 'turn': "From the following text, identify all technical terms and provide a brief explanation for each: 'The API uses OAuth 2.0 for authentication and returns data in JSON format. It supports REST and GraphQL endpoints.'"},
    {'id': 145, 'category': 'extraction', 'turn': "Identify and categorize all the verbs in the following sentence by their tense: 'She has been studying all day, will finish her homework later, and was planning to go for a walk.'"},
    # stem
    {'id': 146, 'category': 'stem',       'turn': 'Explain the process of photosynthesis in detail, including the light-dependent and light-independent reactions.'},
    {'id': 147, 'category': 'stem',       'turn': 'Explain the theory of general relativity and how it differs from Newtonian gravity.'},
    {'id': 148, 'category': 'stem',       'turn': 'Describe the structure of DNA and explain how it stores genetic information.'},
    {'id': 149, 'category': 'stem',       'turn': 'Explain how vaccines work and why some people experience side effects.'},
    {'id': 150, 'category': 'stem',       'turn': 'Describe the water cycle and explain how human activities impact it.'},
    # humanities
    {'id': 151, 'category': 'humanities', 'turn': 'Analyze the causes of World War I, focusing on the underlying tensions that led to the conflict.'},
    {'id': 152, 'category': 'humanities', 'turn': 'Explain the philosophical concept of existentialism and its main proponents.'},
    {'id': 153, 'category': 'humanities', 'turn': 'Discuss the impact of the Industrial Revolution on society, economy, and culture.'},
    {'id': 154, 'category': 'humanities', 'turn': 'Compare and contrast the economic systems of capitalism and socialism.'},
    {'id': 155, 'category': 'humanities', 'turn': "Analyze the themes of power and corruption in Shakespeare's 'Macbeth'."},
]
print(f'Loaded {len(MT_BENCH_QUESTIONS)} questions across {len(set(q["category"] for q in MT_BENCH_QUESTIONS))} categories')

In [ ]:
# Cell 5: Load model
print(f'Loading {MODEL_ID} ...')
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if DEVICE == 'cuda' else torch.float32,
    device_map=DEVICE,
    attn_implementation='flash_attention_2' if (DEVICE == 'cuda' and FLASH_ATTN) else 'eager',
)
model.train(False)  # inference mode

print(f'Model loaded in {time.time()-t0:.1f}s')
if DEVICE == 'cuda':
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM used: {used:.1f} / {total:.0f} GB')

In [ ]:
# Cell 6: QUICK TEST — single coding question, token-by-token table
# Run this first to verify everything works (~2 min)

TEST_QUESTION = 'Write a Python function to implement quicksort.'

# Generate response
messages = [{'role': 'user', 'content': TEST_QUESTION}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Generated response:')
print(generated[:500])

# Measure lookahead on the generated text
gen_inputs = tokenizer(generated, return_tensors='pt').to(DEVICE)
gen_ids = gen_inputs['input_ids'][0]

with torch.no_grad():
    logits = model(**gen_inputs).logits[0]

draft_tokens = logits[:-1].argmax(dim=-1)
actual_tokens = gen_ids[1:]
matches = (draft_tokens == actual_tokens).cpu().numpy()

print(f'\nToken-by-token match table (first 40 tokens):')
print(f'{"i":>4}  {"actual":>14}  {"draft":>14}  {"match"}')
print('-' * 50)
for i in range(min(40, len(matches))):
    actual = repr(tokenizer.decode([int(actual_tokens[i])]))
    draft  = repr(tokenizer.decode([int(draft_tokens[i])]))
    mark = 'OK' if matches[i] else '--'
    print(f'{i:>4}  {actual:>14}  {draft:>14}  {mark}')

match_rate = matches.mean()
print(f'\nMatch rate: {match_rate*100:.1f}%  ({matches.sum()}/{len(matches)} tokens)')

# Run lengths
run_lengths = np.zeros(len(matches), dtype=int)
for i in range(len(matches)-1, -1, -1):
    if matches[i]:
        run_lengths[i] = 1 + (run_lengths[i+1] if i+1 < len(matches) else 0)
expected_free = float((run_lengths * matches).sum() / max(1, matches.sum()))
print(f'Expected free tokens/step: {expected_free:.2f}')
print(f'Theoretical speedup: {1+expected_free:.2f}x')
print('\nQuick test passed!')

In [ ]:
# Cell 7: Core analysis functions

def classify_token(token_str):
    if not token_str.strip():
        return 'whitespace'
    if all(c in '.,;:!?()[]{}"\'-' for c in token_str.strip()):
        return 'punct'
    if token_str.startswith(' ') or token_str.startswith('\u0120'):
        return 'space_word'
    return 'subword_cont'


def measure_lookahead(text):
    """
    Measure how often logits[i] correctly predicts input_ids[i+1].
    Returns dict with match_rate, expected_free_tokens, and per-type breakdown.
    """
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048).to(DEVICE)
    ids = inputs['input_ids'][0]
    if ids.shape[0] < 3:
        return None

    with torch.no_grad():
        logits = model(**inputs).logits[0]

    draft_tokens = logits[:-1].argmax(dim=-1)
    actual_tokens = ids[1:]
    matches = (draft_tokens == actual_tokens).cpu().numpy()
    actual_np = actual_tokens.cpu().numpy()

    run_lengths = np.zeros(len(matches), dtype=int)
    for i in range(len(matches)-1, -1, -1):
        if matches[i]:
            run_lengths[i] = 1 + (run_lengths[i+1] if i+1 < len(matches) else 0)

    type_matches = defaultdict(list)
    for i in range(len(matches)):
        tok = tokenizer.decode([int(actual_np[i])])
        type_matches[classify_token(tok)].append(bool(matches[i]))

    match_rate = float(matches.mean())
    expected_free = float((run_lengths * matches).sum() / max(1, matches.sum()))

    return {
        'seq_len': int(ids.shape[0]),
        'match_rate': match_rate,
        'expected_free_tokens': expected_free,
        'token_type_stats': {
            k: {'match_rate': float(np.mean(v)), 'count': len(v)}
            for k, v in type_matches.items()
        },
    }


def generate_response(question):
    messages = [{'role': 'user', 'content': question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    )

print('Functions defined.')

In [ ]:
# Cell 8: Run benchmark
questions = MT_BENCH_QUESTIONS[:NUM_QUESTIONS]
all_results = []
category_results = defaultdict(list)

print(f'Running {len(questions)} questions...')
print(f'{"#":>3}  {"Cat":12s}  {"Match%":>7s}  {"E[free]":>8s}')
print('-' * 40)

t0 = time.time()
for idx, q in enumerate(questions):
    generated = generate_response(q['turn'])
    stats = measure_lookahead(generated)
    if stats is None:
        continue

    result = {
        'id': q['id'],
        'category': q['category'],
        'question': q['turn'],
        **stats,
    }
    all_results.append(result)
    category_results[q['category']].append(stats['match_rate'])

    print(f"{idx+1:>3}  {q['category']:12s}  {stats['match_rate']*100:6.1f}%  {stats['expected_free_tokens']:8.2f}")

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.0f}s')

In [ ]:
# Cell 9: Summary table + comparison to published baselines
all_rates = [r['match_rate'] for r in all_results]
all_free  = [r['expected_free_tokens'] for r in all_results]

print('RESULTS BY CATEGORY')
print(f'{"Category":15s}  {"Match%":>8s}  {"N":>4s}')
print('-' * 35)
for cat, rates in sorted(category_results.items()):
    print(f'{cat:15s}  {np.mean(rates)*100:7.1f}%  {len(rates):>4d}')
print('-' * 35)
print(f'{"Overall":15s}  {np.mean(all_rates)*100:7.1f}%  {len(all_results):>4d}')

print(f'\nExpected free tokens/step: {np.mean(all_free):.2f}')
print(f'Theoretical speedup:       {1 + np.mean(all_free):.2f}x')

# Comparison to published baselines (approximate, from EAGLE-3 paper)
print('\nComparison to published baselines (Vicuna-7B on MT-Bench):')
print(f'{"Method":30s}  {"Speedup":>10s}  {"Training":>10s}')
print('-' * 58)
print(f'{"EAGLE-3 (reported)":30s}  {"3.05x":>10s}  {"Required":>10s}')
print(f'{"Medusa (reported)":30s}  {"2.83x":>10s}  {"Required":>10s}')
print(f'{"Lookahead Decoding":30s}  {"1.8x":>10s}  {"None":>10s}')
print(f'{f"This work ({MODEL_SIZE})": 30s}  {f"{1+np.mean(all_free):.2f}x":>10s}  {"None":>10s}')

In [ ]:
# Cell 10: Visualization (4 panels)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Lookahead Logit Acceptance Rate — {MODEL_ID}', fontsize=13, fontweight='bold')

# Panel 1: Match rate by category
ax = axes[0, 0]
cats = sorted(category_results.keys())
means = [np.mean(category_results[c]) * 100 for c in cats]
bars = ax.barh(cats, means, color='steelblue')
ax.set_xlabel('Match rate (%)')
ax.set_title('Match rate by category')
ax.axvline(np.mean(all_rates) * 100, color='red', linestyle='--', label=f'Overall {np.mean(all_rates)*100:.1f}%')
ax.legend()
for bar, val in zip(bars, means):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)

# Panel 2: Expected free tokens per step by category
ax = axes[0, 1]
cat_free = {c: np.mean([r['expected_free_tokens'] for r in all_results if r['category'] == c]) for c in cats}
ax.barh(list(cat_free.keys()), list(cat_free.values()), color='darkorange')
ax.set_xlabel('Expected free tokens / step')
ax.set_title('Expected free tokens by category')
ax.axvline(np.mean(all_free), color='red', linestyle='--', label=f'Overall {np.mean(all_free):.2f}')
ax.legend()

# Panel 3: Run-length distribution (histogram of consecutive match runs)
ax = axes[1, 0]
all_run_lengths = []
for r in all_results:
    # Recompute run-length distribution from match_rate (approximate)
    # Geometric distribution approximation: p = match_rate
    pass  # placeholder — actual run lengths would come from raw data

# Use geometric approximation
p = np.mean(all_rates)
simulated_runs = np.random.geometric(1 - p, size=10000)
ax.hist(simulated_runs[simulated_runs <= 20], bins=range(1, 22), density=True,
        color='green', alpha=0.7, edgecolor='black')
ax.set_xlabel('Run length (consecutive free tokens)')
ax.set_ylabel('Frequency')
ax.set_title(f'Run-length distribution (geometric approx, p={p:.2f})')

# Panel 4: Token type breakdown
ax = axes[1, 1]
type_agg = defaultdict(list)
for r in all_results:
    for tok_type, s in r['token_type_stats'].items():
        type_agg[tok_type].extend([s['match_rate']] * s['count'])
tok_types = sorted(type_agg.keys())
tok_means = [np.mean(type_agg[t]) * 100 for t in tok_types]
tok_counts = [len(type_agg[t]) for t in tok_types]
bars = ax.bar(tok_types, tok_means, color='purple', alpha=0.7)
ax.set_ylabel('Match rate (%)')
ax.set_title('Match rate by token type')
for bar, val, cnt in zip(bars, tok_means, tok_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%\n(n={cnt})', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('lookahead_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved lookahead_results.png')

In [ ]:
# Cell 11: Save results + download
model_short = MODEL_ID.split('/')[-1]
output_json = f'lookahead_{model_short}_{RUN_MODE}.json'

summary = {
    'model': MODEL_ID,
    'run_mode': RUN_MODE,
    'num_questions': len(all_results),
    'overall_match_rate': float(np.mean(all_rates)),
    'mean_expected_free_tokens': float(np.mean(all_free)),
    'theoretical_speedup': float(1 + np.mean(all_free)),
    'by_category': {
        cat: {'match_rate': float(np.mean(rates)), 'n': len(rates)}
        for cat, rates in category_results.items()
    },
    'results': all_results,
}

with open(output_json, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved {output_json}')

# Download in Colab
try:
    from google.colab import files
    files.download(output_json)
    files.download('lookahead_results.png')
    print('Download started.')
except ImportError:
    print('Not in Colab — files saved locally.')